<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.14 Local Agents II

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 1. Introduction

In [`05.13_Local_Agents_I.ipynb`](./05.13_Local_Agents_I.ipynb) you built one agent. This notebook takes the two steps that the industry took next. First, we hire a **team**: several models with different roles checking each other's work. Then we point the whole idea at the activity where agents have changed daily practice the most: **writing code**, with your Raspberry Pi as the engine behind VS Code.

Before starting, connect the Raspberry Pi and open the SSH tunnel from [`05.13_Local_Agents_I.ipynb`](./05.13_Local_Agents_I.ipynb), Section 2. For the VS Code part in Section 5 you will additionally install one extension.

### 2. Multi-Agent Systems: Why Teams Beat Heroes

A single model answering in one pass has no chance to reconsider. The cheapest quality upgrade in applied AI is therefore not a bigger model, but a **second look**. Multi-agent systems structure that second look in a few patterns:

| Pattern | Idea | You have already met it |
|:--------|:-----|:------------------------|
| **Pipeline** | each agent transforms the previous agent's output | the training pipeline stages in [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb) |
| **Writer–critic** | one drafts, one reviews against criteria, the first revises | peer review: this section |
| **Orchestrator–specialists** | a lead agent splits the task and delegates | the *ensemble of experts* idea from [`05.07_Introduction_to_Local_LLMs.ipynb`](./05.07_Introduction_to_Local_LLMs.ipynb), Section 4.2 |
| **Debate / voting** | independent agents answer, then compare or vote | self-consistency in [`05.12_Understanding_Reasoning_Models.ipynb`](./05.12_Understanding_Reasoning_Models.ipynb), Section 5.1 |

We build the writer–critic pair, with a twist that matters in practice: the two roles use **different model families**. `gemma3:1b-it-qat` drafts while `qwen3:1.7b` reviews. Different training data means different blind spots, the same reason review panels are not staffed with clones of the author.

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"    # through the SSH tunnel from Local Agents I

try:
    version = requests.get(f"{OLLAMA_URL}/api/version", timeout=3).json()["version"]
    PI_ONLINE = True
    print(f"✅ connected to Ollama {version} through the tunnel")
except requests.exceptions.RequestException:
    PI_ONLINE = False
    print(f"❌ nothing answered at {OLLAMA_URL} - connect the Pi and open the tunnel first.")

CALL_LOG = []                        # every model call files its receipt here

def ask(model, prompt, temperature=0.6, max_tokens=260, no_think=False):
    """One plain completion from one model on the Pi - logged like a billable call."""
    payload = {"model": model, "prompt": prompt, "stream": False,
               "options": {"temperature": temperature, "seed": 7, "num_predict": max_tokens}}
    if no_think:
        payload["think"] = False       # qwen3 would otherwise reason at length first
    data = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=600).json()
    if "error" in data:
        raise RuntimeError(f"Ollama said: {data['error']}")
    CALL_LOG.append({"model": model,
                     "seconds": round(data.get("total_duration", 0) / 1e9, 1),
                     "new tokens": data.get("eval_count", 0)})
    return data["response"].strip()

In [ ]:
WRITER_MODEL = "gemma3:1b-it-qat"
CRITIC_MODEL = "qwen3:1.7b"

BRIEF = ("Recommend either Wi-Fi or 5G connectivity for the card-payment terminal of a "
         "food truck that changes location daily. Exactly three sentences, written for "
         "the truck's owner, who is not technical.")

if PI_ONLINE:
    draft = ask(WRITER_MODEL, f"You are a consultant. {BRIEF}")
    print(f"DRAFT ({WRITER_MODEL})\n{draft}\n")

    critique = ask(CRITIC_MODEL, no_think=True, prompt=(
        "You are a strict reviewer. Assess the draft below against this brief and rubric.\n"
        f"Brief: {BRIEF}\n"
        "Rubric: (1) technically sound reasoning, (2) understandable for a non-technical "
        "reader, (3) exactly three sentences with a clear recommendation.\n"
        "Give at most three numbered improvement points. Do not rewrite the text.\n\n"
        f"Draft:\n{draft}"))
    print(f"CRITIQUE ({CRITIC_MODEL})\n{critique}\n")

    revision = ask(WRITER_MODEL, (
        f"You are a consultant. Your task was: {BRIEF}\n\n"
        f"Your draft:\n{draft}\n\nReviewer feedback:\n{critique}\n\n"
        "Write the improved final version. Output only the three sentences."))
    print(f"REVISION ({WRITER_MODEL})\n{revision}\n")

    import pandas as pd
    bill = pd.DataFrame(CALL_LOG)
    print("THE BILL")
    print(bill.to_string(index=False))
    print(f"one reviewed paragraph cost {bill['seconds'].sum():.0f} s "
          f"and {bill['new tokens'].sum()} generated tokens")
else:
    print("❌ Pi not connected - rerun the connection cell first.")

### 3. Reading the Team's Work

Put the draft and the revision side by side and judge like the client would: did the second look *pay for itself*? Often the critic catches exactly the things small writers get wrong: sentence-count violations, jargon, a recommendation that never commits. Sometimes the critique is generic and the revision worse. That is a real finding too and you will report either outcome in the questions.

Notice what made this a *team* rather than one longer prompt. The critic saw the draft **as an object to judge**, not as its own words to defend. Models, like people, review others more sharply than themselves. And the roles were played by different model families, so the reviewer's blind spots are not copies of the author's. However, the cost is real: the run printed seconds and generated tokens for all three calls. Quality through review is a dial you pay to turn, exactly like the thinking budget in [`05.12_Understanding_Reasoning_Models.ipynb`](./05.12_Understanding_Reasoning_Models.ipynb).

### 4. Vibe Coding: The Agent Gets a Keyboard

In early 2025 Andrej Karpathy gave a name to a practice that was quietly taking over software work: **vibe coding**, describing what you want in natural language, letting the model write the code, running it, and steering with follow-ups instead of editing every line yourself [01]. Between the autocomplete you may know from office software and the fully autonomous coding agents of the frontier labs lies a whole spectrum:

$$
\text{autocomplete} \;\rightarrow\; \text{chat about code} \;\rightarrow\; \textbf{vibe coding} \;\rightarrow\; \text{autonomous coding agents}
$$

The frontier tools run on data-center models. Ours will run on the Raspberry Pi, and that is the experiment: after [`05.08_Making_Models_Smaller.ipynb`](./05.08_Making_Models_Smaller.ipynb) you know precisely *what* was traded away to fit a coding model into 2 GB. Today you feel that trade through other markers: private, offline, free per token, and noticeably less brilliant. Before you start, skim the ground rules in [`00.4_Vibe_Coding_Best_Practices.ipynb`](../00_Intro/00.4_Vibe_Coding_Best_Practices.ipynb); they apply doubly when the model is small.

### 5. Plugging the Pi into VS Code

The tunnel from [`05.13_Local_Agents_I.ipynb`](./05.13_Local_Agents_I.ipynb) is about to come in handy again: because the Pi's Ollama already appears at `localhost:11434`, *any* tool built for a local Ollama now works with the Pi unchanged as the Pi is **proxied** into your laptop. We use [Continue](https://www.continue.dev), an open-source VS Code extension for chat-with-your-code and inline edits:

1. Keep the tunnel terminal from Section 2 open.
2. In VS Code, install the **Continue** extension from the marketplace.
3. Open Continue's configuration and register the Pi's models (the file format may evolve, see the [Continue documentation](https://docs.continue.dev) if the snippet below does not match your version):

```yaml
models:
  - name: qwen3 on the Pi
    provider: ollama
    model: qwen3:1.7b
    apiBase: http://localhost:11434
    roles: [chat, edit]
```

4. Open the Continue panel, select *qwen3 on the Pi* and say hello. The reply is generated by the machine on your desk. Watch `htop` on the Pi to see it work.

> If Continue is not an option on your machine, no harm done: Ollama also speaks the de-facto standard OpenAI API on the same port [04], so most AI coding tools can be pointed at `http://localhost:11434/v1`, and the exercise below works in the plain Continue chat panel or even over `ollama run` in the SSH session.

### 6. The Task: Vibe-Code a Wi-Fi Dashboard

<div class="alert alert-block alert-info">

Create an empty file `dashboard.html` in VS Code and build it **without writing code yourself**, only by instructing your Pi-powered assistant. Work in stages, and after each stage open the file in a browser and check before prompting on:

1. *"Create a single-file HTML page titled 'Site Survey' that shows five rooms and their Wi-Fi signal strength in dBm as horizontal bars."*
2. *"Color bars red when the signal is weaker than −70 dBm, green otherwise."*
3. *"Sort the rooms from strongest to weakest signal."*
4. *"Add a button that re-randomizes each signal by a few dBm and updates the bars."*

Keep a short diary as you go: what each prompt produced, what was wrong, and what you said to fix it. The diary is your evidence for the questions, so pay attention to the *feel* of the tokens-per-second, because that is the size-versus-quality trade of [`05.08_Making_Models_Smaller.ipynb`](./05.08_Making_Models_Smaller.ipynb) measured by your own patience.
</div>

### 7. From Pair Programmer to Engineering Team

Close the loop on today's two ideas: the frontier coding tools you read about are exactly the combination you just practiced: **multi-agent systems pointed at code**. Under the hood of a modern autonomous coding agent, a planner breaks down the request, coder agents edit files and run tests through tools, and reviewer agents critique the diff before a human ever sees it: writer–critic, orchestrator, and pipeline, industrialized [02][03]. The difference between your Pi setup and those systems is the same one as in many other notebooks: scale and polish, not technique.

### Further Reading and Literature

[01] Karpathy, A. (2025). **The original "vibe coding" post.** X (formerly Twitter), February 2025. [Post](https://x.com/karpathy/status/1886192184808149383)

[02] Wu, Q., Bansal, G., Zhang, J., et al. (2023). **AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.** arXiv. [Paper](https://arxiv.org/abs/2308.08155)

[03] Anthropic (2025). **How we built our multi-agent research system.** Engineering blog on orchestrator and specialist agents in production. [Article](https://www.anthropic.com/engineering/built-multi-agent-research-system)

[04] Ollama (2024). **OpenAI compatibility.** Why most AI tools can talk to your Pi. [Blog](https://ollama.com/blog/openai-compatibility)

----
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. Many questions ask for concrete values or outputs from your own notebook run; generic answers without run-specific evidence may receive reduced credit. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**Paste your draft and your revision. Quote one concrete critique point and show where (or whether) the revision addressed it. Did the second look pay for itself?** Answer:

(add your answer here)

**Why does it help that writer and critic come from different model families? What is the analogous rule in human peer review?** Answer:

(add your answer here)

**Report THE BILL from your run: seconds and tokens per call, and the totals. For which two of your own regular tasks would you pay this threefold cost, and for which would you not? Justify briefly.** Answer:

(add your answer here)

**Swap the roles (qwen3 writes, gemma3 critiques) or make the critic stricter, and rerun. Report one difference in the outcome.** Answer:

(add your answer here)

**From your vibe-coding diary: quote the prompt that worked best and the moment the model failed most badly. How did you recover: new prompt, manual edit, or restart?** Answer:

(add your answer here)

**Describe the felt tokens-per-second of the Pi assistant compared with a cloud chatbot you know. Using the evidence from [`05.08_Making_Models_Smaller.ipynb`](./05.08_Making_Models_Smaller.ipynb), what exactly buys the cloud tool its advantage?** Answer:

(add your answer here)

**A colleague proposes replacing your company's cloud coding assistant with local models for confidentiality reasons. Based on today's experience: what would you support, what would you push back on, and what pilot would you run to decide?** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)